In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

In [ ]:
# fig02_3axis_concept.png - 3-axis embedding concept
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

origin = np.array([0, 0, 0])
axis_len = 1.5

# Draw Re axis (red) - SigLIP2, 1152d
ax.quiver(*origin, axis_len, 0, 0, color='red', arrow_length_ratio=0.12,
          linewidth=2.5, label='Re (SigLIP2, 1152d)')
ax.text(axis_len + 0.15, 0, 0, 'Re\n(SigLIP2, 1152d)', color='red',
        fontsize=10, fontweight='bold', ha='left')

# Draw Im axis (blue) - BGE-M3, 1024d
ax.quiver(*origin, 0, axis_len, 0, color='blue', arrow_length_ratio=0.12,
          linewidth=2.5, label='Im (BGE-M3, 1024d)')
ax.text(0, axis_len + 0.15, 0, 'Im\n(BGE-M3, 1024d)', color='blue',
        fontsize=10, fontweight='bold', ha='center')

# Draw Z axis (green) - DINOv2, 1024d
ax.quiver(*origin, 0, 0, axis_len, color='green', arrow_length_ratio=0.12,
          linewidth=2.5, label='Z (DINOv2, 1024d)')
ax.text(0, 0, axis_len + 0.15, 'Z\n(DINOv2, 1024d)', color='green',
        fontsize=10, fontweight='bold', ha='center')

# Sample query point and document point
query_pt = np.array([0.9, 1.1, 0.7])
doc_pt = np.array([1.1, 0.8, 1.0])

ax.scatter(*query_pt, color='orange', s=120, zorder=5, label='Query point')
ax.text(query_pt[0] + 0.05, query_pt[1] + 0.05, query_pt[2] + 0.05,
        'Query', fontsize=9, color='orange', fontweight='bold')

ax.scatter(*doc_pt, color='purple', s=120, marker='^', zorder=5, label='Document point')
ax.text(doc_pt[0] + 0.05, doc_pt[1] + 0.05, doc_pt[2] + 0.05,
        'Document', fontsize=9, color='purple', fontweight='bold')

# Draw dashed line between query and document
xs = [query_pt[0], doc_pt[0]]
ys = [query_pt[1], doc_pt[1]]
zs = [query_pt[2], doc_pt[2]]
ax.plot(xs, ys, zs, 'k--', linewidth=1.2, alpha=0.6)

# Draw dotted projection lines to axes for query point
ax.plot([query_pt[0], query_pt[0]], [query_pt[1], query_pt[1]], [0, query_pt[2]],
        'gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.plot([query_pt[0], query_pt[0]], [0, query_pt[1]], [0, 0],
        'gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.plot([0, query_pt[0]], [query_pt[1], query_pt[1]], [0, 0],
        'gray', linestyle=':', linewidth=0.8, alpha=0.5)

# Formula annotation using ax.text2D
formula = r'score = $\sqrt{A^2 + (\alpha \cdot B)^2 + (\beta \cdot C)^2}$' + '\n' + r'$\alpha=0.4,\ \beta=0.2$'
ax.text2D(0.02, 0.02, formula, transform=ax.transAxes,
          fontsize=11, color='black',
          bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow',
                    edgecolor='gray', alpha=0.9))

ax.set_xlim(0, axis_len + 0.4)
ax.set_ylim(0, axis_len + 0.4)
ax.set_zlim(0, axis_len + 0.4)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_zlabel('')
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

ax.set_title('Complex-Hermitian 3축 임베딩 구조', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=9, framealpha=0.85)
ax.view_init(elev=20, azim=225)

output_path = os.path.join(os.getcwd(), 'fig02_3axis_concept.png')
plt.tight_layout()
plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {output_path}')

In [ ]:
# fig02_domain_encoder_matrix.png - 도메인별 인코더 활성화 매트릭스 (코드 검증 반영)
import matplotlib.colors as mcolors

domains = ['Doc', 'Img', 'Movie', 'Rec', 'BGM']
encoders = ['Re\n(SigLIP2)', 'Im\n(BGE-M3)', 'Z\n(DINOv2)', 'STT\n(Whisper)', 'Caption\n(BLIP)']

# 코드 기반 실제 활성화 매트릭스:
# unified_engine.py: Doc → cache_doc_page_Re/Im/Z.npy (3축 모두 존재)
# unified_engine.py: Img → cache_img_Re_siglip2/Im_e5cap/Z_dinov2.npy (3축 + BLIP)
# unified_engine.py: Movie → _build_av_entry (Re 필수, Im/Z fallback) + STT + Caption
# unified_engine.py: Music(Rec) → _build_av_entry (Re 필수, Im fallback=Re, Z fallback=zeros) + STT
# BGM: 동일 AV 구조 + STT
# 값: 1.0=활성, 0.5=조건부(fallback), 0.0=비활성
matrix = np.array([
    [1, 1, 1, 0, 0],     # Doc: Re+Im+Z (페이지 이미지 렌더링)
    [1, 1, 1, 0, 1],     # Img: Re+Im+Z + BLIP 캡션
    [1, 1, 1, 1, 1],     # Movie: 전체 활성
    [1, 1, 0.5, 1, 0],   # Rec: Re+Im+STT, Z=조건부(zeros fallback)
    [1, 1, 0.5, 1, 0],   # BGM: Re+Im+STT, Z=조건부(zeros fallback)
], dtype=float)

domain_color_list = [COLORS[d] for d in domains]

fig, ax = plt.subplots(figsize=(11, 6.5))

n_rows, n_cols = matrix.shape

for r in range(n_rows):
    for c in range(n_cols):
        val = matrix[r, c]
        if val >= 1.0:
            cell_color = domain_color_list[r]
            symbol = '\u2714'
            text_color = 'white'
            alpha = 0.85
        elif val == 0.5:
            # 조건부: 연한 색 + 물결표
            cell_color = domain_color_list[r]
            symbol = '~'
            text_color = 'white'
            alpha = 0.4
        else:
            cell_color = 'white'
            symbol = '\u2717'
            text_color = '#AAAAAA'
            alpha = 1.0

        rect = mpatches.FancyBboxPatch(
            (c - 0.45, r - 0.45), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            linewidth=1.2,
            edgecolor='#888888',
            facecolor=cell_color,
            alpha=alpha
        )
        ax.add_patch(rect)
        ax.text(c, r, symbol, ha='center', va='center',
                fontsize=18, color=text_color, fontweight='bold')

ax.set_xlim(-0.5, n_cols - 0.5)
ax.set_ylim(-0.5, n_rows - 0.5)
ax.set_xticks(range(n_cols))
ax.set_xticklabels(encoders, fontsize=11)
ax.set_yticks(range(n_rows))
ax.set_yticklabels(domains, fontsize=12, fontweight='bold')

for tick_label, color in zip(ax.get_yticklabels(), domain_color_list):
    tick_label.set_color(color)

ax.invert_yaxis()
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.tick_params(axis='both', which='both', length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title('도메인별 인코더 활성화 매트릭스', fontsize=14, fontweight='bold', pad=40)

# Legend: 활성 / 조건부 / 비활성
legend_patches = [
    mpatches.Patch(facecolor='#555555', edgecolor='gray', label='\u2714 활성', alpha=0.85),
    mpatches.Patch(facecolor='#555555', edgecolor='gray', label='~ 조건부 (fallback zeros)', alpha=0.4),
    mpatches.Patch(facecolor='white', edgecolor='gray', label='\u2717 비활성'),
]
ax.legend(handles=legend_patches, loc='lower right',
          fontsize=10, title='범례', title_fontsize=10,
          framealpha=0.9, bbox_to_anchor=(1.15, 0))

output_path2 = os.path.join(os.getcwd(), 'fig02_domain_encoder_matrix.png')
plt.tight_layout()
plt.savefig(output_path2, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {output_path2}')

In [ ]:
# fig02_domain_axis_usage.png - 도메인별 3축 활용 및 Null 스코어 분포 (코드 검증 반영)
import json

with open('../publication/paper/null_hist_real.json', 'r') as f:
    null_data = json.load(f)

with open('../Data/embedded_DB/Bgm/calibration.json', 'r') as f:
    bgm_cal = json.load(f)

with open('../publication/paper/_phase_ridge_results.json', 'r') as f:
    ridge_data = json.load(f)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('도메인별 Tri-CHEF 축 활용 및 Null 스코어 분포', fontsize=16, fontweight='bold', y=0.98)

# 코드 검증 결과 반영:
# Doc: cache_doc_page_Re/Im/Z.npy 모두 존재 → 3축 전체 (페이지 이미지 렌더링)
# Img: cache_img_Re/Im/Z.npy → 3축 전체
# Movie: _build_av_entry Re+Im+Z(fallback) → 3축 전체
# Rec(music): _build_av_entry Re+Im, Z=zeros fallback → 2D+조건부
# BGM: 동일 AV 구조 → 2D+조건부
domain_configs = {
    'Doc':   {'axes_active': ['Re','Im','Z'], 'axes_cond': [],
              'dim': '3D', 'color': COLORS['Doc'],
              'mu': null_data['domains']['doc_page']['mu'],
              'sigma': null_data['domains']['doc_page']['sigma'],
              'note': '페이지 이미지 렌더링'},
    'Img':   {'axes_active': ['Re','Im','Z'], 'axes_cond': [],
              'dim': '3D', 'color': COLORS['Img'],
              'mu': null_data['domains']['image']['mu'],
              'sigma': null_data['domains']['image']['sigma'],
              'note': 'BLIP 3단계 캡션'},
    'Movie': {'axes_active': ['Re','Im'], 'axes_cond': ['Z'],
              'dim': '3D', 'color': COLORS['Movie'],
              'mu': null_data['domains']['movie']['mu'],
              'sigma': null_data['domains']['movie']['sigma'],
              'note': 'STT + 프레임 캡션'},
    'Rec':   {'axes_active': ['Re','Im'], 'axes_cond': ['Z'],
              'dim': '2D~3D', 'color': COLORS['Rec'],
              'mu': null_data['domains']['music']['mu'],
              'sigma': null_data['domains']['music']['sigma'],
              'note': 'STT 기반, Z fallback'},
    'BGM':   {'axes_active': ['Re','Im'], 'axes_cond': ['Z'],
              'dim': '2D~3D', 'color': COLORS['BGM'],
              'mu': bgm_cal['mu_null'],
              'sigma': bgm_cal['sigma_null'],
              'note': 'STT 기반, Z fallback'},
}

for idx, (domain, cfg) in enumerate(domain_configs.items()):
    row, col = divmod(idx, 3)
    ax = axes[row][col]

    axis_labels = ['Re', 'Im', 'Z']
    bar_vals = []
    bar_colors = []
    bar_hatches = []
    for a in axis_labels:
        if a in cfg['axes_active']:
            bar_vals.append(1.0)
            bar_colors.append(cfg['color'])
            bar_hatches.append('')
        elif a in cfg.get('axes_cond', []):
            bar_vals.append(0.7)
            bar_colors.append(cfg['color'])
            bar_hatches.append('///')
        else:
            bar_vals.append(0.0)
            bar_colors.append('#E0E0E0')
            bar_hatches.append('')

    x_labels = ['Re\n(SigLIP2)', 'Im\n(BGE-M3)', 'Z\n(DINOv2)']
    bars = ax.bar(range(3), bar_vals, color=bar_colors, edgecolor='gray',
                  linewidth=1.2, width=0.6)
    for b, h, v in zip(bars, bar_hatches, bar_vals):
        b.set_hatch(h)
        b.set_alpha(0.9 if v >= 1.0 else (0.5 if v > 0 else 0.2))

    ax.set_xticks(range(3))
    ax.set_xticklabels(x_labels, fontsize=9)
    ax.set_ylim(-0.1, 1.7)
    ax.set_yticks([0, 0.7, 1])
    ax.set_yticklabels(['OFF', '조건부', 'ON'], fontsize=8)

    # Scoring dimension badge
    ax.text(1, 1.45, f'{cfg["dim"]} Scoring', ha='center', fontsize=11,
            fontweight='bold', color=cfg['color'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor=cfg['color'],
                      alpha=0.15, edgecolor=cfg['color']))

    # Null distribution params
    ax.text(1, 1.2, f'$\\mu_{{null}}$={cfg["mu"]:.3f}  $\\sigma$={cfg["sigma"]:.3f}',
            ha='center', fontsize=8.5, color='#444444')

    # Note
    ax.text(1, -0.07, cfg['note'], ha='center', fontsize=8, color='#777777', style='italic')

    ax.set_title(domain, fontsize=13, fontweight='bold', color=cfg['color'], pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 6th subplot: summary
axes[1][2].axis('off')
axes[1][2].text(0.5, 0.6,
    'score = $\\sqrt{A^2 + (\\alpha B)^2 + (\\beta C)^2}$\n'
    '$\\alpha=0.4,\\ \\beta=0.2$\n\n'
    'Doc/Img: 3축 전체 활성\n'
    'Movie: Re+Im 필수, Z 조건부\n'
    'Rec/BGM: Re+Im 필수, Z fallback\n\n'
    '■ 활성  ▨ 조건부  □ 비활성',
    transform=axes[1][2].transAxes, ha='center', va='center',
    fontsize=11, linespacing=1.6,
    bbox=dict(boxstyle='round,pad=0.6', facecolor='lightyellow',
              edgecolor='gray', alpha=0.9))

plt.tight_layout(rect=[0, 0, 1, 0.95])
output_path3 = os.path.join(os.getcwd(), 'fig02_domain_axis_usage.png')
plt.savefig(output_path3, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {output_path3}')